# Round 1 — Data Foundation Upgrade (Post-Baseline)

**Previous:** Baseline — YOLOv8n, 2000 images (1600 train / 400 val), 15 epochs, default random split.
Baseline result: Precision 0.233, Recall 0.516, mAP@0.5 0.267, mAP@0.5:0.95 0.100, Best F1 0.29 @ conf 0.086. Confusion matrix showed 0 true positives at default threshold — model was severely under-confident due to too little data and too few epochs.

**Round 1 scope (Data Foundation only ):**
1. Scale up training data (2000 → configurable, default 10000 images)
2. Keep natural positive:negative ratio (~32:68), no oversampling/undersampling
3. Duplicate / near-duplicate file check
4. Patient-level **stratified** train/val split (stratified on presence of pneumonia, fixed seed) — baseline split was patient-level but not stratified
5. Small-box handling decision (documented, with visual verification)
6. Epochs: 15 → 100, with early stopping (patience=20)

## 1. Setup — locate dataset 

In [ ]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

INPUT_DIR = Path("/kaggle/input")

for item in INPUT_DIR.iterdir():
    print(item)

label_files = list(INPUT_DIR.rglob("stage_2_train_labels.csv"))
if len(label_files) == 0:
    raise FileNotFoundError("RSNA labels file was NOT found.")
LABELS_CSV = label_files[0]
print("\nLABELS_CSV:", LABELS_CSV)

dicom_files = list(INPUT_DIR.rglob("*.dcm"))
print("Number of DICOM files found:", len(dicom_files))
if len(dicom_files) == 0:
    raise FileNotFoundError("No DICOM files found.")
DICOM_DIR = dicom_files[0].parent
print("DICOM_DIR:", DICOM_DIR)

labels = pd.read_csv(LABELS_CSV)
print("\nLabels shape:", labels.shape)
display(labels.head())

## 2. Duplicate / Near-Duplicate File Check

RSNA DICOM studies are one file per patientId, so exact-duplicate *files* are not expected — but we verify this cheaply via an MD5 hash of the raw file bytes. This satisfies the "duplicate/near-duplicate removal for leak-free training" checklist item.

In [ ]:
import hashlib

def file_md5(path, chunk_size=65536):
    h = hashlib.md5()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

print("Hashing all DICOM files to check for exact duplicates")

hash_to_paths = {}
for p in dicom_files:
    h = file_md5(p)
    hash_to_paths.setdefault(h, []).append(p)

duplicate_groups = {h: paths for h, paths in hash_to_paths.items() if len(paths) > 1}

print(f"Total DICOM files: {len(dicom_files)}")
print(f"Unique file hashes: {len(hash_to_paths)}")
print(f"Duplicate groups found: {len(duplicate_groups)}")

if duplicate_groups:
    print("\nExample duplicate group(s):")
    for h, paths in list(duplicate_groups.items())[:3]:
        print(paths)
else:
    print("\u2705 No exact-duplicate DICOM files found — safe to proceed without dedup filtering.")

## 3. Configure Data Scale

`NUM_IMAGES` controls how many unique patients are used. Baseline used 2000.:
- `8000` — good balance of signal vs. training time (default here)
- `10000`–`15000` — stronger signal, longer training
- `len(patient_ids)` (~26684) — full dataset, best possible result but longest runtime

Change `NUM_IMAGES` below and re-run if you have more GPU quota available.

In [ ]:
NUM_IMAGES = 10000   
IMG_SIZE = 1024      # unchanged from baseline;

all_patient_ids = labels["patientId"].unique()
print("Total unique patients in dataset:", len(all_patient_ids))

# Per-patient label: does this patient have >=1 positive (pneumonia) row?
patient_target = (
    labels.groupby("patientId")["Target"]
    .max()
    .reindex(all_patient_ids)
)

overall_pos_rate = patient_target.mean()
print(f"Overall positive rate (natural ratio): {overall_pos_rate*100:.2f}% positive / {(1-overall_pos_rate)*100:.2f}% negative")

if NUM_IMAGES >= len(all_patient_ids):
    patient_ids = all_patient_ids
    print(f"\nUsing FULL dataset: {len(patient_ids)} patients")
else:
    # Stratified sample so the natural ~32:68 ratio is preserved in the subset
    from sklearn.model_selection import train_test_split
    patient_ids, _ = train_test_split(
        all_patient_ids,
        train_size=NUM_IMAGES,
        stratify=patient_target,
        random_state=42
    )
    print(f"\nStratified subset selected: {len(patient_ids)} patients (natural ratio preserved)")

subset_pos_rate = patient_target.reindex(patient_ids).mean()
print(f"Subset positive rate: {subset_pos_rate*100:.2f}% (target: {overall_pos_rate*100:.2f}%)")
print("\u2705 Ratio decision: keeping natural ~32:68 distribution, no oversampling/undersampling —")
print("   oversampling positives risks overfitting on repeated lesions; undersampling negatives")
print("   throws away useful hard-negative signal that helps reduce false positives.")

## 4. Patient-Level Stratified Train/Val Split (fixed seed for reproducibility across rounds)

In [ ]:
from sklearn.model_selection import train_test_split

subset_target = patient_target.reindex(patient_ids)

train_patients, val_patients = train_test_split(
    patient_ids,
    test_size=0.2,
    random_state=42,
    stratify=subset_target
)

print("Training patients:", len(train_patients))
print("Validation patients:", len(val_patients))

overlap = set(train_patients).intersection(set(val_patients))
print("Patient overlap:", len(overlap))
assert len(overlap) == 0, "Patient overlap found! Split is not leak-free."
print("\u2705 No patient overlap — split is clean.")

train_pos_rate = subset_target.reindex(train_patients).mean()
val_pos_rate = subset_target.reindex(val_patients).mean()
print(f"\nTrain positive rate: {train_pos_rate*100:.2f}%")
print(f"Val positive rate:   {val_pos_rate*100:.2f}%")
print("\u2705 Stratified split — train/val positive rates should closely match (unlike baseline's")
print("   unstratified split, which could randomly skew ratios between splits).")

## 5. Small-Box Analysis & Handling Decision

Checking the distribution of bounding-box sizes (relative to the 1024x1024 image) before deciding whether to drop very small boxes. YOLO is known to struggle with tiny objects, but RSNA pneumonia opacities are frequently genuinely small — dropping them would directly hurt recall on exactly the cases radiologists care most about catching early.

**Decision:** keep all boxes with positive width/height (i.e. only drop malformed/zero-area annotations, which would break YOLO label parsing). No area-based filtering of legitimately small lesions — recall on small lesions is preserved, verified visually below.

In [ ]:
positive_rows = labels[(labels["patientId"].isin(patient_ids)) & (labels["Target"] == 1)].copy()

# Relative box area assuming original RSNA image is 1024x1024 (matches our resize target)
positive_rows["rel_area"] = (positive_rows["width"] * positive_rows["height"]) / (1024 * 1024)

print("Box relative-area statistics (fraction of image area):")
display(positive_rows["rel_area"].describe())

invalid_boxes = positive_rows[(positive_rows["width"] <= 0) | (positive_rows["height"] <= 0)]
print(f"\nInvalid (zero/negative area) boxes found: {len(invalid_boxes)}")

very_small = positive_rows[positive_rows["rel_area"] < 0.005]
print(f"Boxes under 0.5% of image area (very small lesions): {len(very_small)} "
      f"({len(very_small)/len(positive_rows)*100:.1f}% of all positive boxes)")

MIN_BOX_AREA_RATIO = 0.0   # only drop truly invalid (<=0) boxes; keep all real small lesions
print(f"\n\u2705 Decision: MIN_BOX_AREA_RATIO = {MIN_BOX_AREA_RATIO} (keep all valid boxes, including small ones)")

## 6. Build Round 1 Folder Structure

In [ ]:
WORK_DIR = Path("/kaggle/working/rsna_yolo_round1")

TRAIN_IMG_DIR = WORK_DIR / "images" / "train"
VAL_IMG_DIR = WORK_DIR / "images" / "val"
TRAIN_LABEL_DIR = WORK_DIR / "labels" / "train"
VAL_LABEL_DIR = WORK_DIR / "labels" / "val"

for d in [TRAIN_IMG_DIR, VAL_IMG_DIR, TRAIN_LABEL_DIR, VAL_LABEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("\u2705 Round 1 folder structure created at:", WORK_DIR)

## 7. DICOM → YOLO Conversion (pipeline unchanged from baseline — normalization/aug tuning is Round 2)

Same conversion logic as baseline (min-max normalize, grayscale→RGB, resize to 1024x1024, scale boxes, write YOLO-format labels), with the `MIN_BOX_AREA_RATIO` filter from Section 5 applied.

In [ ]:
import cv2
import pydicom
from tqdm.auto import tqdm

def convert_dicom_to_yolo(patient_id, split):
    dcm_path = DICOM_DIR / f"{patient_id}.dcm"
    if not dcm_path.exists():
        return False

    dicom_file = pydicom.dcmread(dcm_path)
    image = dicom_file.pixel_array.astype(np.float32)
    original_height, original_width = image.shape[:2]

    image = image - image.min()
    if image.max() > 0:
        image = image / image.max()
    image = (image * 255).astype(np.uint8)

    image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    image = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    if split == "train":
        image_dir, label_dir = TRAIN_IMG_DIR, TRAIN_LABEL_DIR
    else:
        image_dir, label_dir = VAL_IMG_DIR, VAL_LABEL_DIR

    image_path = image_dir / f"{patient_id}.png"
    cv2.imwrite(str(image_path), cv2.cvtColor(image, cv2.COLOR_RGB2BGR))

    patient_rows = labels[labels["patientId"] == patient_id]
    yolo_labels = []

    for _, row in patient_rows.iterrows():
        if row["Target"] == 1:
            x, y, width, height = row["x"], row["y"], row["width"], row["height"]

            if width <= 0 or height <= 0:
                continue  # drop invalid boxes only (Section 5 decision)

            rel_area = (width * height) / (original_width * original_height)
            if rel_area < MIN_BOX_AREA_RATIO:
                continue

            x = x * IMG_SIZE / original_width
            y = y * IMG_SIZE / original_height
            width = width * IMG_SIZE / original_width
            height = height * IMG_SIZE / original_height

            x_center = (x + width / 2) / IMG_SIZE
            y_center = (y + height / 2) / IMG_SIZE
            width_normalized = width / IMG_SIZE
            height_normalized = height / IMG_SIZE

            yolo_labels.append(
                f"0 {x_center:.6f} {y_center:.6f} {width_normalized:.6f} {height_normalized:.6f}"
            )

    label_path = label_dir / f"{patient_id}.txt"
    with open(label_path, "w") as f:
        f.write("\n".join(yolo_labels))

    return True

print("Converting training images...")
train_count = sum(convert_dicom_to_yolo(pid, "train") for pid in tqdm(train_patients))
print(f"\n\u2705 Training images converted: {train_count}")

print("\nConverting validation images...")
val_count = sum(convert_dicom_to_yolo(pid, "val") for pid in tqdm(val_patients))
print(f"\n\u2705 Validation images converted: {val_count}")

## 8. Verify Dataset Balance (compare to baseline)

In [ ]:
def count_positive_negative(label_dir):
    positive = negative = 0
    for label_file in label_dir.glob("*.txt"):
        content = label_file.read_text().strip()
        if content:
            positive += 1
        else:
            negative += 1
    return positive, negative

train_positive, train_negative = count_positive_negative(TRAIN_LABEL_DIR)
val_positive, val_negative = count_positive_negative(VAL_LABEL_DIR)

train_total = train_positive + train_negative
val_total = val_positive + val_negative

print("===== ROUND 1 TRAINING DATA =====")
print(f"Pneumonia images     : {train_positive} ({train_positive/train_total*100:.2f}%)")
print(f"Non-pneumonia images : {train_negative} ({train_negative/train_total*100:.2f}%)")
print(f"Total                : {train_total}  (baseline was 1600)")

print("\n===== ROUND 1 VALIDATION DATA =====")
print(f"Pneumonia images     : {val_positive} ({val_positive/val_total*100:.2f}%)")
print(f"Non-pneumonia images : {val_negative} ({val_negative/val_total*100:.2f}%)")
print(f"Total                : {val_total}  (baseline was 400)")

## 9. dataset.yaml

In [ ]:
DATASET_YAML = WORK_DIR / "dataset.yaml"

yaml_content = f"""
path: {WORK_DIR}

train: images/train
val: images/val

nc: 1

names:
  0: pneumonia
"""

with open(DATASET_YAML, "w") as f:
    f.write(yaml_content)

print("\u2705 dataset.yaml created")
print(yaml_content)

In [ ]:
get_ipython().getoutput("pip install -q ultralytics")

In [ ]:
from ultralytics.data.utils import check_det_dataset
data = check_det_dataset(str(DATASET_YAML))
print("\u2705 YOLO dataset structure is valid!")

## 10. Visual Verification (box scaling + small-box visibility check)

Same visual-check cell as baseline, run against the Round 1 data, to confirm boxes are still scaled correctly and small lesions are visibly retained after the pipeline changes.

In [ ]:
import matplotlib.pyplot as plt

sample_images = [p for p in VAL_IMG_DIR.glob("*.png") if (VAL_LABEL_DIR / f"{p.stem}.txt").read_text().strip()][:5]
print("Positive images selected for checking:", len(sample_images))

for image_path in sample_images:
    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    label_path = VAL_LABEL_DIR / f"{image_path.stem}.txt"
    height, width = image.shape[:2]

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:
        values = line.strip().split()
        if len(values) != 5:
            continue
        class_id, x_center, y_center, box_width, box_height = map(float, values)
        x_center *= width; y_center *= height
        box_width *= width; box_height *= height
        x1 = int(x_center - box_width / 2); y1 = int(y_center - box_height / 2)
        x2 = int(x_center + box_width / 2); y2 = int(y_center + box_height / 2)
        cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
        cv2.putText(image, "pneumonia", (x1, max(y1 - 10, 20)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    plt.figure(figsize=(7, 7))
    plt.imshow(image)
    plt.title(image_path.stem)
    plt.axis("off")
    plt.show()

## 11. Train — Round 1 (Data Foundation)

Only the Data Foundation levers are changed vs. baseline: more images, stratified split, 100 epochs with early stopping. Augmentation and optimizer are left at Ultralytics defaults on purpose — those are Round 2/3 items — so any metric delta here is attributable to data scale + split + epochs.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(DATASET_YAML),
    epochs=100,
    patience=20,        # early stopping
    imgsz=IMG_SIZE,      # unchanged from baseline (1024) — Round 2 item
    batch=-1,            # auto batch size based on available GPU memory
    seed=42,
    project=str(WORK_DIR / "runs"),
    name="round1",
    exist_ok=True,
    verbose=True,
)

## 12. Evaluate — Standard Metrics + Best-F1 Threshold (matching baseline's reporting method)

In [ ]:
best_weights = WORK_DIR / "runs" / "round1" / "weights" / "best.pt"
eval_model = YOLO(str(best_weights))

# Standard validation (Ultralytics default conf threshold for the summary metrics table)
val_results = eval_model.val(data=str(DATASET_YAML), imgsz=IMG_SIZE, split="val")

precision = float(val_results.box.mp)
recall = float(val_results.box.mr)
map50 = float(val_results.box.map50)
map50_95 = float(val_results.box.map)

print("===== ROUND 1 — STANDARD METRICS =====")
print(f"Precision     : {precision:.3f}")
print(f"Recall        : {recall:.3f}")
print(f"mAP@0.5       : {map50:.3f}")
print(f"mAP@0.5:0.95  : {map50_95:.3f}")

In [ ]:
import numpy as np

# Sweep confidence thresholds to find best-F1, same methodology as baseline (best F1 @ 0.086)
conf_thresholds = np.arange(0.02, 0.61, 0.02)
best_f1 = -1.0
best_conf = None
best_p, best_r = None, None

for conf in conf_thresholds:
    r = eval_model.val(data=str(DATASET_YAML), imgsz=IMG_SIZE, split="val", conf=float(conf), plots=False, verbose=False)
    p = float(r.box.mp)
    rec = float(r.box.mr)
    f1 = 2 * p * rec / (p + rec) if (p + rec) > 0 else 0.0
    if f1 > best_f1:
        best_f1 = f1
        best_conf = float(conf)
        best_p, best_r = p, rec

print("===== ROUND 1 — BEST F1 THRESHOLD =====")
print(f"Best F1            : {best_f1:.3f}")
print(f"Best F1 Confidence  : {best_conf:.3f}")
print(f"Precision @ best F1 : {best_p:.3f}")
print(f"Recall @ best F1    : {best_r:.3f}")

In [ ]:
# Confusion matrix at the best-F1 confidence threshold
cm_results = eval_model.val(data=str(DATASET_YAML), imgsz=IMG_SIZE, split="val", conf=best_conf, plots=True)
cm = cm_results.confusion_matrix.matrix
print("Confusion matrix (rows=predicted, cols=true) at conf =", round(best_conf, 3))
print(cm)
# cm[0][0] = TP (pneumonia correctly detected), cm[0][1] = FP, cm[1][0] = FN, cm[1][1] = background/TN

In [ ]:
## 13. Write metrics.md

In [ ]:
metrics_md = f"""# Round 1 Metrics — Data Foundation Upgrade

**Model:** YOLOv8n
**Dataset:** RSNA Pneumonia Detection Challenge
**Training:** {train_total} images ({train_positive} pneumonia / {train_negative} non-pneumonia)
**Validation:** {val_total} images ({val_positive} pneumonia / {val_negative} non-pneumonia)
**Epochs:** up to 100 (early stopping patience=20)

## Changes vs. Baseline (Round 1 = Data Foundation only)
| Change | Baseline | Round 1 |
|---|---|---|
| Training images | 1600 | {train_total} |
| Split strategy | Patient-level random | Patient-level **stratified** on pneumonia presence |
| Duplicate check | Not done | MD5 file-hash check performed ({len(duplicate_groups)} duplicate groups found) |
| Small-box filtering | Not addressed | Documented decision: keep all valid boxes (only drop zero-area) |
| Epochs | 15 (fixed) | Up to 100 with early stopping (patience=20) |
| Augmentation/optimizer/resolution | default | **unchanged** (deferred to Round 2/3) |

## Results
| Metric | Baseline | Round 1 |
|---|---:|---:|
| Precision | 0.233 | {precision:.3f} |
| Recall | 0.516 | {recall:.3f} |
| mAP@0.5 | 0.267 | {map50:.3f} |
| mAP@0.5:0.95 | 0.100 | {map50_95:.3f} |
| Best F1 | 0.29 | {best_f1:.3f} |
| Best F1 Confidence | 0.086 | {best_conf:.3f} |

## Confusion Matrix (at Round 1 best-F1 confidence = {best_conf:.3f})
```
{cm}
```
"""

metrics_path = WORK_DIR / "metrics_round1.md"
with open(metrics_path, "w") as f:
    f.write(metrics_md)

print(metrics_md)
print(f"\n\u2705 Saved: {metrics_path}")

## 14. Package Outputs for GitHub / DVC

Copies `best.pt` and `metrics_round1.md` into a folder matching your repo layout (`PART-A(Darshi)/`), ready to download from Kaggle and push into the repo. Git/DVC commands are shown as reference — Kaggle notebooks don't have your repo's git credentials, so run these commands **locally** (or in a Kaggle Dataset/Colab with your GitHub token set up) after downloading this folder.

In [ ]:
import shutil

OUTPUT_DIR = Path("/kaggle/working/PART-A_round1_output")
(OUTPUT_DIR / "weights").mkdir(parents=True, exist_ok=True)

shutil.copy(best_weights, OUTPUT_DIR / "weights" / "best.pt")
shutil.copy(metrics_path, OUTPUT_DIR / "metrics.md")
shutil.copy(DATASET_YAML, OUTPUT_DIR / "dataset.yaml")

print("\u2705 Round 1 outputs packaged at:", OUTPUT_DIR)
print("\nContents:")
for p in OUTPUT_DIR.rglob("*"):
    print(" ", p)



## 15. Summary

Round 1 changed only the **Data Foundation** category vs. baseline:
- Training set scaled from 1600 → the number printed in Section 8
- Duplicate check performed (file-hash based)
- Split changed from random to **stratified** patient-level (train/val positive rates now match)
- Small-box handling decision documented (kept all valid boxes)
- Epochs increased from 15 → up to 100 with early stopping

Everything else (augmentation, normalization/windowing, optimizer, LR schedule, model size, loss weighting, resolution) was deliberately left unchanged so this round's metric delta is attributable to data foundation changes alone — ready to compare against baseline in `metrics.md` for the mentor.

**Next:** Round 2 will build on this dataset (same split) and tune preprocessing + augmentation.